# 💾 SQLite Database Management (Lost & Found Matcher)
This notebook manages the persistent SQLite database (`lost_found.db`):
- Schema definition (`reports` table)
- Seeding sample realistic reports
- CRUD operations (Create, Read, Update, Delete)


In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

DB_PATH = "lost_found.db"

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT sqlite_version()")
print("SQLite Version:", cursor.fetchone()[0])
conn.close()


SQLite Version: 3.50.4


### Step 1: Create Schema and Seed Initial Reports


In [2]:
conn = get_connection()
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_type TEXT NOT NULL CHECK(report_type IN ('lost', 'found')),
        item_name TEXT NOT NULL,
        category TEXT NOT NULL,
        color TEXT NOT NULL,
        location TEXT NOT NULL,
        report_date TEXT NOT NULL,
        description TEXT NOT NULL,
        image_path TEXT,
        contact_info TEXT,
        status TEXT NOT NULL DEFAULT 'open',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()

cursor.execute("SELECT COUNT(*) FROM reports")
count = cursor.fetchone()[0]
print(f"Current reports count in database: {count}")
conn.close()


Current reports count in database: 17


### Step 2: Querying Reports with Pandas


In [3]:
conn = get_connection()
df_db = pd.read_sql_query("SELECT id, report_type, item_name, category, color, location, report_date, status FROM reports ORDER BY id DESC LIMIT 10", conn)
conn.close()

print(f"Retrieved {len(df_db)} reports:")
df_db


Retrieved 10 reports:


### Step 3: Database Summary Metrics


In [4]:
conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM reports WHERE report_type = 'lost'")
total_lost = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM reports WHERE report_type = 'found'")
total_found = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM reports WHERE status = 'resolved'")
total_resolved = cursor.fetchone()[0]
conn.close()

print("Summary Statistics:")
print(f"  Total Lost Items : {total_lost}")
print(f"  Total Found Items: {total_found}")
print(f"  Resolved Cases   : {total_resolved}")


Summary Statistics:
  Total Lost Items : 8
  Total Found Items: 9
  Resolved Cases   : 0
